# Solving 2d Burgers Equation by Using P2C2Net

## Background Introduction

### Overview

**P2C2Net (PDE-Preserved Coarse Correction Network)** is a novel neural network architecture designed to efficiently solve spatiotemporal partial differential equations (PDEs) on coarse mesh grids with limited training data. Original paper is [P2C2Net: PDE-Preserved Coarse Correction Network for Efficient Prediction of Spatiotemporal Dynamics](https://arxiv.org/pdf/2411.00040).

![model architecture](./images/model_architecture.png)

As shown in the figure above, the model consists of two synergistic modules: (1) a trainable PDE block that learns to update the coarse solution (i.e., the system state), based on a high-order numerical scheme with boundary condition encoding, and (2) a neural network block that consistently corrects the solution on the fly. In particular, the model adopts a learnable symmetric Conv filter, with weights shared over the entire model, to accurately estimate the spatial derivatives of PDE based on the neural-corrected system state.

### Burger's Equation

The Burgers’ equation is a nonlinear PDE that models the propagation and reflection of shock waves. It is widely used in fluid mechanics, nonlinear acoustics, gas dynamics, and other fields.

$$
\frac{\partial \mathbf{u}}{\partial t}=\nu \nabla^2 \mathbf{u}-\mathbf{u}\cdot \nabla \mathbf{u}, t\in [0,T], x\in [0,1]^2
$$

Periodic boundaries are used to avoid non-physical reflections/errors caused by artificially specified computational domains, and are suitable for unbounded domain problems as well as periodic physical structures/phenomena. The core requirement is that physical quantities satisfy numerical equality and continuous derivatives on the "corresponding boundaries" of the computational domain.

$$
\mathbf{u}(\mathbf{x}_1, t)=\mathbf{u}(\mathbf{x}_2, t), \nabla\mathbf{u}(\mathbf{x}_1, t)=\nabla\mathbf{u}(\mathbf{x}_2, t)
$$

Where $\mathbf{x}_1\in \partial \Omega_1, \mathbf{x}_2\in \partial \Omega_2$ are the periodic corresponding points on the boundaries.

### Problem Descriptions

In this project, we focus on solving the **2D Burgers’ equation** efficiently using P2C2Net.

$$
\mathbf{u}_t \mapsto \mathbf{u}(\cdot, t+1)
$$

## Model Implementation

### Hardware Requirements

NPU memory>32G

### MindSpore & MindScience Version

mindspore>=2.5.0
mindscience==0.8.0

### Installation

1. Ensure that the correct versions of MindSpore and MindScience are installed in the environment;
2. Additional python packages, such as numpy、pandas、sympy、matplotlib, should be installed advancedly.
3. Clone the MindScience repository or obtain the codes directly from [MindFlow/applications/data_mechanism_fusion/p2c2net](https://atomgit.com/mindspore-lab/mindscience/tree/master/MindFlow/applications/data_mechanism_fusion/p2c2net);

### Dataset

Download train and test dataset: [MindFlow/applications/data_mechanism_fusion/p2c2net/src/data_gen.py](https://atomgit.com/mindspore-lab/mindscience/blob/master/MindFlow/applications/data_mechanism_fusion/p2c2net/src/data_gen.py).

### Coding

The specific process for solving this problem using MindFlow is as follows:

1. Dataset construction.
2. Model construction.
3. Model training.
4. Model inference and visualization.

In [ ]:
import json
import signal
import time

import mindspore as ms
import mindspore.common.dtype as mstype
import numpy as np
from mindspore import Tensor, context, nn, set_seed
from mindspore.amp import auto_mixed_precision
from mindspore.dataset import GeneratorDataset

The following `src` pacakage can be downloaded in [applications/data_mechanism_fusion/p2c2net/src](https://atomgit.com/mindspore-lab/mindscience/tree/master/MindFlow/applications/data_mechanism_fusion/p2c2net/src).

In [ ]:
from src.data import (
    MyDataset,
    UnitGaussianNormalizer,
    ensure_directories,
    evl_error,
    generate_dataset,
    get_data,
    plot_loss,
)
from src.data_gen import createdata
from src.model import RCNN, P2N2Net

In [ ]:
# environment parameters setup
set_seed(12345)
np.random.seed(12345)

data_dir = "data"
model_dir = "model"
model_name = "P2C2Net_burgers"
config_dir = "config"
result_dir = "result"
loss_dir = "loss"
error_dir = "error"

In [ ]:
args = {}
args["experiment_directory"]=""
args["config_filename"]="burgers.json"
args["continue_from"]=None
args["mode"]="PYNATIVE"
args["device_target"]="Ascend"
args["device_id"]=0
args["test_stage"]=True

In [ ]:
# set mindspore context
ms.context.set_context(
    mode=context.GRAPH_MODE if args["mode"].upper().startswith("GRAPH") else context.PYNATIVE_MODE)
if args["device_target"].upper() != "CPU":
    ms.set_device(args["device_target"], args["device_id"])
use_ascend = context.get_context(attr_key='device_target') == "Ascend"
compute_dtype = mstype.float32
ms.set_recursion_limit(99999999)

In [ ]:
# load config file
config_path = os.path.join(args["experiment_directory"], config_dir, args["config_filename"])
with open(config_path, 'r', encoding='utf-8') as f:
    burgers_config = json.load(f)

#### Dataset construction

The training and test datasets need to be manually generated.

In this case, the training and test datasets are generated with reference to the dataset settings described by Qi Wang in the paper [P2C2Net: PDE-Preserved Coarse Correction Network for Efficient Prediction of Spatiotemporal Dynamics](https://arxiv.org/pdf/2411.00040). The specific settings are as follows:

Based on periodic boundary conditions, the initial condition $\mathbf{u}_0(x,y)$ satisfying the following distribution is generated:

$$
\mathbf{u}_0(x,y)=A(i,j)\sin(2\pi*(a(i)x+b(j)y))+B(i,j)\cos(2\pi*(a(i)x+b(j)y))+c
$$

Where $i,j$ correspond to the grid indices of $x,y$ in the spatial domain, respectively.

The data is generated using the high-order Finite Difference (FD) method (Runge-Kutta 4, RK4) with a time step of 1e-3. The final data records the solution every t = 1 time unit, advancing for 1400 time steps. The spatial domain is $[0,1]^2$, and all data is generated on a 100×100 grid before being downsampled to a 25×25 grid. In this case, the viscosity coefficient $\nu=1e−3$, with 10 samples in the training set and 5 samples in the test set.


In [ ]:
# generate data with different random seeds
if not os.path.exists("./data"): os.mkdir("./data")
for seed in range(1, 16):
    createdata(seed)

In [ ]:
num = burgers_config["num_data"]
train_win = burgers_config["train_window"]
timesteps = burgers_config["timesteps"]
batch_size = burgers_config["batch_size"]
down = burgers_config["down"]
nolap = burgers_config["nolap"]
drop = burgers_config["drop"]
normalization = burgers_config["normalization"]
pretrain_iters = burgers_config["pretrain_iters"]
pretrain_window = burgers_config["pretrain_window"]
n_train = burgers_config["num_train"]
n_test = burgers_config["num_test"]

In [ ]:
def signal_handler(sig, frame):  # pylint: disable=unused-argument
    """Handle interrupt signal for graceful shutdown."""

    print("Stopping early...")
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)

In [ ]:
# [batch, ic, steps, uv, nx, ny]
all_data = get_data(
    args["experiment_directory"],
    num,
    down
)

normalizer = None
if normalization:
    normalizer = UnitGaussianNormalizer(
        x = all_data,
        normalization_dim = (0, 1, 3, 4)
    )
    all_data = normalizer.encode(all_data)

train_data = all_data[: n_train]
test_data = all_data[-n_test:]

In [ ]:
print("*****************trainingStage****************")

all_data_set = generate_dataset(
    train_data[:, :timesteps],
    train_win=train_win,
    icnum=n_train,
    nolap=nolap
)
print("all data set shape: ", all_data_set.shape)
# [400, 5, 2, 26, 26] [ic, times, uv, nx, ny]
train_dataset = GeneratorDataset(
    source = MyDataset(all_data_set),
    column_names = ['data', 'labels'],
    shuffle=False
)

train_loader = train_dataset.batch(batch_size, drop)

if pretrain_iters != 0 and pretrain_window != 0:
    pretrain_data_set = generate_dataset(
        train_data[:, :timesteps],
        train_win=pretrain_window,
        icnum=n_train,
        nolap=nolap
    )
    pretrain_dataset = GeneratorDataset(
    source = MyDataset(pretrain_data_set),
    column_names = ['data', 'labels'],
    shuffle=False
    )
    pretrain_loader = pretrain_dataset.batch(batch_size, drop)
else:
    pretrain_loader = None

#### Model construction

The RCNN, serving as the backbone model to be trained, is embedded into the P2N2Net model framework for training.

In [ ]:
milestone_num = burgers_config["milestone_num"]
epochs = burgers_config['epochs']
weight_decay = burgers_config["weight_decay"]
save_every = burgers_config["save_every"]
gamma = burgers_config['gamma']
lr = burgers_config['learning_rate']
model_config = burgers_config["model"]
size = burgers_config["size"]
resolution = size // down
deno = 1 / size * down
delta_t = burgers_config["delta_t"]
modes = model_config['modes']
depths = model_config['depths']
h_channels = model_config['hidden_channels']
p_channels = model_config['projection_channels']

In [ ]:
# learning rate, start_epoch, etc. variables setup
if milestone_num is not None:
    milestones = [(epochs // milestone_num) * (i + 1) for i in range(milestone_num)]
    learning_rates = [gamma**i * float(lr) for i in range(milestone_num)]
    learning_rate = nn.piecewise_constant_lr(milestones, learning_rates)
else:
    learning_rate = burgers_config['learning_rate']

start_epoch = 1

In [ ]:
model = RCNN(
    **model_config,
    delta_t=delta_t, compute_dtype=compute_dtype,
    resolution=resolution, deno=deno,
    time_steps=train_win, use_ascend=use_ascend
)
print("Model")
print(model)

model_param = f"e{epochs}_k{modes}_d{depths}_l{h_channels}_p{p_channels}"
result_save_dir = os.path.join(args["experiment_directory"], result_dir)
model_save_dir = os.path.join(result_save_dir, model_dir, model_param, model_name)
loss_save_dir = os.path.join(result_save_dir, loss_dir, model_param)
ensure_directories(result_save_dir, model_save_dir, loss_save_dir)

if args["continue_from"] is not None:
    ms.load_checkpoint(model_save_dir + f'checkpoints_{args["continue_from"]}.ckpt', model)
    start_epoch = args["continue_from"]
    print(f"training continum from checkpoints {args['continue_from']}")

net = P2N2Net(
    model, learning_rate,
    weight_decay, use_ascend
)

#### Model training

With MindSpore >= 2.7.0, neural networks can be trained using the functional programming paradigm.

In [ ]:
model.set_train(True)
if pretrain_loader:
    print("pretrain start")
    model.steps = pretrain_window
    for i in range(pretrain_iters):
        print("pretrain ", i)
        pretrain_batch_count = 0
        pretrain_epoch_loss = 0
        for batch_data in pretrain_loader.create_dict_iterator():
            # [ic, uv, nx, ny] , [step, uv, nx, ny]
            pretrain_batch_loss = net(batch_data)
            pretrain_epoch_loss += pretrain_batch_loss
            pretrain_batch_count += 1
            print(f'batch {pretrain_batch_count} loss {pretrain_batch_loss}')
        print("Pretraining epoch Loss: ", pretrain_epoch_loss / pretrain_batch_count)

In [ ]:
# test FNO
from mindscience.models import FNO2D

data = Tensor(np.ones([2, 3, 128, 128]), mstype.float32)
net = FNO2D(in_channels=3, out_channels=3, n_modes=[20, 20], resolutions=[128, 128], data_format="channels_first")
out = net(data)
print(data.shape, out.shape)

In [ ]:
start = time.time()
train_loss_list = []
print("training start")
model.steps = train_win
for epoch in range(start_epoch, 1 + epochs):
    print(f'epoch {epoch} start')
    epoch_start = time.time()
    epoch_loss = 0
    batch_count = 0
    for batch_data in train_loader.create_dict_iterator():
        # [ic, uv, nx, ny] , [step, uv, nx, ny]
        batch_loss = net(batch_data)
        epoch_loss += batch_loss
        batch_count += 1
        print(f'batch {batch_count} loss {batch_loss}')

    epoch_loss = epoch_loss / batch_count
    train_loss_list.append(epoch_loss)
    epoch_end = time.time()
    epoch_time = epoch_end - epoch_start
    total_time = epoch_end - start
    print("training epoch Loss: ", epoch_loss, "epoch Time: ", epoch_time, "total Time: ", total_time)

    np.savetxt(loss_save_dir + "/train_loss.txt", train_loss_list)
    if epoch % save_every == 0 or epoch ==  epochs:
        ms.save_checkpoint(
            save_obj=model,
            ckpt_file_name = model_save_dir + f"_checkpoints_{epoch}.ckpt"
        )
        ms.save_checkpoint(
            save_obj=model,
            ckpt_file_name = model_save_dir + ".ckpt"
        )
plot_loss(train_loss_list, loss_save_dir)
end = time.time()
total_time = end - start
print("training stage end, total time = ", total_time)

#### Model inference and visualization

In [ ]:
def test_stage(
    test_data,
    exp_dir,
    config,
    compute_dtype,
    use_ascend,
    normalizer=None,
    checkpoint=None
):
    '''
    Evaluate the trained model on test data and compute error metrics.
    '''
    model_config = config["model"]
    down = config["down"]
    size = config["size"]
    resolution = size // down
    deno = 1 / size * down
    train_win = config["train_window"]
    inferstep = config["inferstep"]
    delta_t = config["delta_t"]
    epochs = config["epochs"]
    modes = model_config['modes']
    depths = model_config['depths']
    h_channels = model_config['hidden_channels']
    p_channels = model_config['projection_channels']

    model = RCNN(
        **model_config,
        compute_dtype=compute_dtype,
        resolution=resolution,
        deno=deno,
        time_steps=train_win
    )
    model_param = f"e{epochs}_k{modes}_d{depths}_l{h_channels}_p{p_channels}"
    result_save_dir = os.path.join(exp_dir, result_dir)
    model_save_dir = os.path.join(result_save_dir, model_dir, model_param, model_name)
    error_save_dir = os.path.join(result_save_dir, error_dir, model_param)
    ensure_directories(error_save_dir)

    if checkpoint:
        model_save_dir += f"_checkpoints_{checkpoint}"
    if use_ascend:
        auto_mixed_precision(model, 'O1')

    try:
        ms.load_checkpoint(model_save_dir + '.ckpt', model)
        print("Successfully loaded model")
    except Exception as e:
        print(f"Failed to load model: {e}")
        sys.exit(0)

    model.set_train(False)
    evl_error(
        test_data[:, 0: inferstep],
        model,
        inferstep,
        delta_t,
        resolution,
        error_save_dir,
        compute_dtype,
        normalizer,
    )

In [ ]:
if args["teststage"]:
    print("*****************testingStage****************")
    test_stage(
        test_data,
        args["experiment_directory"],
        burgers_config,
        compute_dtype,
        use_ascend,
        normalizer,
        checkpoint=args["continue_from"]
    )